In [ ]:
import os
import glob
import pickle

from langchain_core.documents import Document
from langchain_text_splitters import MarkdownTextSplitter, RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http import models
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
import torch
import gc

from config import Config


In [ ]:
# Load configuration from config.py
QDRANT_URL = Config.QDRANT_URL
QDRANT_API_KEY = Config.QDRANT_API_KEY
OPENAI_API_KEY = Config.OPENAI_API_KEY
os.environ["OPENAI_API_KEY"] = Config.OPENAI_API_KEY
HF_TOKEN = Config.HF_TOKEN
EMBEDDING_MODEL_ID = Config.EMBEDDING_MODEL
EMBEDDING_DIM = Config.EMBEDDING_DIM
BATCH_SIZE = Config.BATCH_SIZE
LLM_MODEL_ID = Config.MODEL_ID
# DATA_FOLDER_PATH and PICKLE_BACKUP_PATH can still use dynamic naming if needed
current_date = datetime.datetime.now().strftime("%Y_%m_%d__%H%M")
COLLECTION_NAME = Config.COLLECTION_NAME
PICKLE_BACKUP_PATH = Config.PICKLE_PATH


In [6]:
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
PICKLE_BACKUP_PATH = "/kaggle/input/datasets/jayawinata/chunked-medical-reports/2026_04_29__1545_medical_documents_enriched.pkl"

In [ ]:
if os.path.exists(PICKLE_BACKUP_PATH):
    with open(PICKLE_BACKUP_PATH, "rb") as f:
        enriched_documents = pickle.load(f)
    print(f"Berhasil memuat {len(enriched_documents)} dokumen dari {PICKLE_BACKUP_PATH}")
else:
    print(f"File {PICKLE_BACKUP_PATH} belum ada.")

Berhasil memuat 7237 dokumen dari /kaggle/input/datasets/jayawinata/chunked-medical-reports/2026_04_29__1545_medical_documents_enriched.pkl


In [16]:
print("\nMemulai proses Embedding & Upload ke Qdrant...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_ID,
    model_kwargs={
        "model_kwargs": {"device_map": "cuda", "quantization_config": bnb_config},
        "trust_remote_code": True,
        "token": HF_TOKEN,
    },
    encode_kwargs={
        "batch_size": BATCH_SIZE,
    }
)

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

if not client.collection_exists(COLLECTION_NAME):
    print(f"   - Membuat collection baru: {COLLECTION_NAME}")
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=EMBEDDING_DIM,
            distance=models.Distance.COSINE
        ),
        metadata={
            "embedding_model": EMBEDDING_MODEL_ID
        }
    )


Memulai proses Embedding & Upload ke Qdrant...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

   - Membuat collection baru: 2026_05_17__1956_enriched_medical_reports


In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
vector_store = QdrantVectorStore.from_documents(
    documents=enriched_documents,
    embedding=embedding_model,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name=COLLECTION_NAME,
    force_recreate=True,
    batch_size=BATCH_SIZE
)

print(f"\nSELESAI! {len(enriched_documents)} chunks berhasil masuk ke Qdrant.")

OutOfMemoryError: CUDA out of memory. Tried to allocate 56.79 GiB. GPU 0 has a total capacity of 14.56 GiB of which 2.67 GiB is free. Including non-PyTorch memory, this process has 11.89 GiB memory in use. Of the allocated memory 11.67 GiB is allocated by PyTorch, and 101.73 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
!nvidia-smi